In [1]:
from shared_utils.catalog_utils import get_catalog
from shared_utils.rt_dates import DATES
from calitp_data_analysis import sql, gcs_pandas
import pandas as pd

g = gcs_pandas.GCSPandas()

In [2]:
# The feed to examine TODO: support multiple
GTFS_DATASET_NAME = "Bay Area 511 SamTrans Schedule"
# Sample dates to look at
dates_values = DATES.values()
print(len(dates_values))

85


In [3]:
# Get feed key for the target feed:
dates_str = "('" + "', '".join(dates_values) + "')"
feed_keys_to_dates = sql.query_sql(
    f"""
    select 
        tu.base64_url as tu_base64_url,
        sched.feed_key as schedule_feed_key,
        sched.date
    from mart_gtfs.fct_daily_schedule_feeds as sched
    left join mart_gtfs.fct_daily_rt_feed_files as tu
        on sched.feed_key = tu.schedule_feed_key
        and sched.date = tu.date
        and tu.feed_type = 'trip_updates'
    where 
        sched.date in {dates_str} 
        and sched.gtfs_dataset_name = '{GTFS_DATASET_NAME}'
    """,

)
feed_keys_to_dates

/home/mrtopsyt/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


,tu_base64_url,schedule_feed_key,date
0,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,4dfefaf518766d1cfc3d254a65e4295e,2026-06-10
1,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,0215a1c49f081dc0a7eb1e118ea91229,2024-12-11
2,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,15b3a87c05331f6d0e0b73cf340bfc5b,2025-02-12
3,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,1d276b06a3c6ff0d661e25035982ead7,2024-03-13
4,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,25c80082b079cb7fab299c7015aa7470,2023-07-12
...,...,...,...
80,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,c1c5d0b862094fbf640e18075b290861,2025-09-24
81,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,c1c5d0b862094fbf640e18075b290861,2025-09-25
82,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,c7ab222b14ca19ad45369c228ac16783,2025-09-23
83,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,c7bc4bc9de15ff685470b0de7578d313,2026-01-14


In [4]:
# Count fct_stop_time_metrics rows per trip updates feed per date
# The dates with a significant number of rows are the dates where both TU and VP based stop times are available
tu_urls = feed_keys_to_dates["tu_base64_url"].dropna().unique()
tu_urls_str = "('" + "', '".join(tu_urls) + "')"

stop_time_metrics_counts = sql.query_sql(
    f"""
    select
        base64_url as tu_base64_url,
        service_date,
        count(*) as n_stop_time_metrics
    from mart_gtfs.fct_stop_time_metrics
    where
        service_date in {dates_str}
        and base64_url in {tu_urls_str}
    group by base64_url, service_date
    order by service_date
    """,
)
stop_time_metrics_counts


/home/mrtopsyt/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


,tu_base64_url,service_date,n_stop_time_metrics
0,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,2025-12-17,59708
1,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,2026-01-14,59560
2,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,2026-04-08,36667
3,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L3RyaXB1cG...,2026-06-10,60858


In [5]:
# Get the VP based stop times
gtfs_analytics_catalog = get_catalog("gtfs_analytics_data")
stop_time_metrics_counts["vp_uri"] = (
    f"{gtfs_analytics_catalog['rt_stop_times']['dir']}{gtfs_analytics_catalog['rt_stop_times']['stage3']}_" 
    + stop_time_metrics_counts["service_date"].astype(str)
    + ".parquet"
)


In [6]:
g.read_parquet(stop_time_metrics_counts['vp_uri'][0]).head()

,trip_instance_key,stop_sequence,shape_array_key,stop_meters,arrival_time
0,00003a69e91a4ef117840d081cdbe355,1,fd6a664f2ca715017d46996e1cf0c8e0,1.613893,2025-12-17 00:18:17
1,00003a69e91a4ef117840d081cdbe355,2,fd6a664f2ca715017d46996e1cf0c8e0,734.456238,2025-12-17 00:20:53
2,00003a69e91a4ef117840d081cdbe355,4,fd6a664f2ca715017d46996e1cf0c8e0,1359.918670,2025-12-17 00:21:28
3,00003a69e91a4ef117840d081cdbe355,5,fd6a664f2ca715017d46996e1cf0c8e0,1772.076525,2025-12-17 00:22:40
4,00003a69e91a4ef117840d081cdbe355,6,fd6a664f2ca715017d46996e1cf0c8e0,2046.929960,2025-12-17 00:23:32


In [7]:
# Outer join VP based and TU based stop times, per date.
# fct_stop_time_metrics has no trip_instance_key, so fct_scheduled_trips bridges
# its (trip_id, schedule_base64_url, service_date) grain to the VP trip_instance_key.
stm_dates = stop_time_metrics_counts["service_date"].astype(str).tolist()
stm_dates_str = "('" + "', '".join(stm_dates) + "')"
stm_urls_str = "('" + "', '".join(stop_time_metrics_counts["tu_base64_url"].unique()) + "')"

# The VP parquets are statewide, so we need this operator's trips to filter them.
# This also keeps trips that VP saw but TU never predicted.
operator_trips = sql.query_sql(
    f"""
    select trip_instance_key, trip_id, service_date
    from mart_gtfs.fct_scheduled_trips
    where
        service_date in {stm_dates_str}
        and name = '{GTFS_DATASET_NAME}'
    """,
)

# One partition-pruned scan for all dates rather than a query per date.
# Literal service_date/base64_url filters on both tables keep BigQuery from
# scanning the full fct_scheduled_trips history (~48GB) via the join predicate.
tu_stop_times = sql.query_sql(
    f"""
    with stm as (
        select service_date, schedule_base64_url, trip_id, stop_id, stop_sequence,
               actual_arrival_pacific, n_predictions
        from mart_gtfs.fct_stop_time_metrics
        where
            service_date in {stm_dates_str}
            and base64_url in {stm_urls_str}
    ),
    sched as (
        select trip_instance_key, trip_id, base64_url, service_date
        from mart_gtfs.fct_scheduled_trips
        where
            service_date in {stm_dates_str}
            and name = '{GTFS_DATASET_NAME}'
    )
    select
        sched.trip_instance_key,
        stm.service_date,
        stm.stop_id,
        stm.stop_sequence,
        stm.actual_arrival_pacific as tu_arrival_time,
        stm.n_predictions
    from stm
    inner join sched
        on sched.service_date = stm.service_date
        and sched.trip_id = stm.trip_id
        and sched.base64_url = stm.schedule_base64_url
    """,
)

/home/mrtopsyt/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(
/home/mrtopsyt/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


In [8]:
tu_stop_times.head()

,trip_instance_key,service_date,stop_id,stop_sequence,tu_arrival_time,n_predictions
0,a75354193c17193e3722d540b2f3e14a,2026-01-14,334084,53,2026-01-14 17:42:05,89
1,a75354193c17193e3722d540b2f3e14a,2026-01-14,334620,54,2026-01-14 17:48:00,90
2,464ffe4d45b99ce827735f2e0f0735a0,2026-01-14,334084,53,2026-01-14 23:11:20,90
3,464ffe4d45b99ce827735f2e0f0735a0,2026-01-14,334620,54,2026-01-14 23:18:00,90
4,ac9fad6be23d178b6d05b30c2ef14f27,2026-01-14,334084,53,2026-01-14 18:07:05,90


In [9]:
# Get matching VP stop times

VP_COLS = ["trip_instance_key", "stop_sequence", "arrival_time", "stop_meters", "shape_array_key"]

# trip_instance_key is unique per trip per service date, so it both filters the
# statewide VP files down to this operator and carries the date back onto the
# joined rows. Pushing the filter into the read gets ~200k rows off GCS instead
# of the ~9.6M statewide rows these four files hold.
trip_dates = operator_trips.set_index("trip_instance_key")["service_date"]

vp_stop_times = g.read_parquet(
    stop_time_metrics_counts["vp_uri"].tolist(),
    columns=VP_COLS,
    filters=[("trip_instance_key", "in", set(trip_dates.index))],
).rename(columns={"arrival_time": "vp_arrival_time"})

vp_tu_stop_times = vp_stop_times.merge(
    tu_stop_times.drop(columns=["service_date"]),
    on=["trip_instance_key", "stop_sequence"],
    how="outer",
    indicator="source",
)
vp_tu_stop_times["source"] = vp_tu_stop_times["source"].cat.rename_categories(
    {"left_only": "vp_only", "right_only": "tu_only"}
)
vp_tu_stop_times["service_date"] = vp_tu_stop_times["trip_instance_key"].map(trip_dates)

print(
    vp_tu_stop_times.groupby(["service_date", "source"], observed=True)
    .size()
    .unstack(fill_value=0)
)
vp_tu_stop_times.head()


source        vp_only  tu_only   both
service_date                         
2025-12-17       1594    10608  49100
2026-01-14       1151    10809  48751
2026-04-08      19911     6742  29925
2026-06-10       1465    13132  47726


,trip_instance_key,stop_sequence,vp_arrival_time,stop_meters,shape_array_key,stop_id,tu_arrival_time,n_predictions,source,service_date
0,00003a69e91a4ef117840d081cdbe355,1,2025-12-17 00:18:17,1.613893,fd6a664f2ca715017d46996e1cf0c8e0,NaN,NaT,NaN,vp_only,2025-12-17
1,00003a69e91a4ef117840d081cdbe355,2,2025-12-17 00:20:53,734.456238,fd6a664f2ca715017d46996e1cf0c8e0,NaN,NaT,NaN,vp_only,2025-12-17
2,00003a69e91a4ef117840d081cdbe355,4,2025-12-17 00:21:28,1359.918670,fd6a664f2ca715017d46996e1cf0c8e0,NaN,NaT,NaN,vp_only,2025-12-17
3,00003a69e91a4ef117840d081cdbe355,5,2025-12-17 00:22:40,1772.076525,fd6a664f2ca715017d46996e1cf0c8e0,NaN,NaT,NaN,vp_only,2025-12-17
4,00003a69e91a4ef117840d081cdbe355,6,2025-12-17 00:23:32,2046.929960,fd6a664f2ca715017d46996e1cf0c8e0,NaN,NaT,NaN,vp_only,2025-12-17


In [ ]:
vp_tu_stop_times.loc[vp_tu_stop_times.source == "tu_only"].head(20)


,trip_instance_key,stop_sequence,vp_arrival_time,stop_meters,shape_array_key,stop_id,tu_arrival_time,n_predictions,source,service_date
70,0008d298e3f42cea9e00cfddb076fdd0,1,NaT,NaN,NaN,341081,2026-06-10 07:18:00,79.0,tu_only,2026-06-10
78,0008d298e3f42cea9e00cfddb076fdd0,9,NaT,NaN,NaN,341956,2026-06-10 07:36:00,90.0,tu_only,2026-06-10
79,000e8ebad1f93d750cc8f215289b4ce1,1,NaT,NaN,NaN,363600,2025-12-17 16:46:00,180.0,tu_only,2025-12-17
82,000e8ebad1f93d750cc8f215289b4ce1,4,NaT,NaN,NaN,363608,2025-12-17 16:49:00,180.0,tu_only,2025-12-17
105,000e8ebad1f93d750cc8f215289b4ce1,25,NaT,NaN,NaN,346119,2025-12-17 17:18:14,90.0,tu_only,2025-12-17
118,000e8ebad1f93d750cc8f215289b4ce1,38,NaT,NaN,NaN,344085,2025-12-17 17:42:02,90.0,tu_only,2025-12-17
119,000e8ebad1f93d750cc8f215289b4ce1,39,NaT,NaN,NaN,344631,2025-12-17 18:17:00,90.0,tu_only,2025-12-17
218,00287d5217fce4451c815fc789e55634,1,NaT,NaN,NaN,332048,2025-12-17 06:16:00,90.0,tu_only,2025-12-17
219,00287d5217fce4451c815fc789e55634,2,NaT,NaN,NaN,331430,2025-12-17 06:16:48,90.0,tu_only,2025-12-17
240,00287d5217fce4451c815fc789e55634,23,NaT,NaN,NaN,332266,2025-12-17 06:40:18,90.0,tu_only,2025-12-17
